In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv(r"C:\Users\jayal\Documents\Marketing_Attribution_Project\Cleaned_attribution_data.csv")

In [3]:
conn = sqlite3.connect(r"C:\Users\jayal\Documents\Marketing_Attribution_Project\attribution.db")

In [4]:
df.to_sql('attribution',conn,if_exists='replace',index=False)
print("Data loaded successfully!")
print(f"Total rows loaded:{len(df)}")


Data loaded successfully!
Total rows loaded:586737


In [13]:
#window function: user journey sequencing
query="""
SELECT cookie,time,channel,row_number() OVER (PARTITION by cookie ORDER by time) as touch_number FROM attribution LIMIT 20;"""
df2 = pd.read_sql_query(query,conn)
df2.head()

,cookie,time,channel,touch_number
0,00000FkCnDfDDf0iC97iC703B,2018-07-03T13:02:11Z,Instagram,1
1,00000FkCnDfDDf0iC97iC703B,2018-07-17T19:15:07Z,Online Display,2
2,00000FkCnDfDDf0iC97iC703B,2018-07-24T15:51:46Z,Online Display,3
3,00000FkCnDfDDf0iC97iC703B,2018-07-29T07:44:51Z,Online Display,4
4,0000nACkD9nFkBBDECD3ki00E,2018-07-03T09:44:57Z,Paid Search,1


In [14]:
#First click attribution
query="""
with journey as(

      select cookie,channel,

	  row_number() over (PARTITION by cookie order by time) as touch_number

	  from attribution

)

select channel,count(*) as first_click_count

from journey

where touch_number=1

group by channel;
"""
result = pd.read_sql_query(query,conn)
result.head()

,channel,first_click_count
0,Facebook,66848
1,Instagram,28618
2,Online Display,34250
3,Online Video,34182
4,Paid Search,76210


In [5]:
#linear attribution
query1="""
with journey as (
   select cookie, channel,
   count(*) over (partition by cookie) as total_touches
   from attribution
)
select channel,
round(sum(1.0/total_touches),2) as linear_attribution_score
from journey
group by channel
order by linear_attribution_score desc;
"""
df3 =  pd.read_sql_query(query1,conn)
print(df3)

          channel  linear_attribution_score
0     Paid Search                  74560.28
1        Facebook                  67311.76
2    Online Video                  35199.74
3  Online Display                  34168.10
4       Instagram                  28868.12


In [ ]:
#last click attribution
query2="""
with journey as (
      select cookie,channel,
	  row_number() over (partition by cookie order by time desc) as touch_number
	  from attribution
)
select channel,count(*) as last_click_count
from journey
where touch_number=1
group by channel
order by last_click_count desc;
"""
df4 = pd